# Image-labels metadata

To ease the interpretation of label images and make them identifiable as label images,
it is helpful to add the [`image-label` metadata](https://ngff.openmicroscopy.org/specifications/0.5/index.html#labels-metadata) to the written label image data.
The {py:class}`ome_zarr.NgffMultiscales` provides an easy entrypoint to pass this metadata when writing label images to disk and to read it back in when loading an image with labels from disk.

Similar to the [previous notebook](advanced:labels:adding_labels), we will start by creating a parent image and a dummy label image, to which we add some metadata.

In [1]:
import numpy as np
from ome_zarr import NgffMultiscales, NgffImage
from skimage.data import binary_blobs
from ome_zarr_models.common.image_label_types import LabelBase, Color, Property

In [2]:
rng = np.random.default_rng(0)
data = rng.poisson(10, size=(64, 64, 64)).astype(np.uint8)

ngff_image = NgffImage(data, axes="zyx", name="image")
ngff_multiscales = NgffMultiscales(image=ngff_image)

In [3]:
labels = NgffImage(
    data=binary_blobs(length=64, volume_fraction=0.1, n_dim=3).astype('int8'),
    axes="zyx",
)
labels_multiscales = NgffMultiscales(image=labels)

## Composing the image-label metadata

In Python context, the `image-label` metadata is a dictionary composed of the following fields:
- `colors`: Consists of the actual integer label value and a list of rgba color values.
- `properties`: Unspecified, per-object key-value pairs. The `label-value` property links the metadata to the actual label value in the label image.
- `source`: The source of the label image. Automatically written by the writer tool.
- `version`: The version of the metadata schema. Automatically written by the writer tool.

In our example (a binary label image), we need only provide the metadata for two kinds of objects (i.e., the background and the foreground):

In [4]:
colors = [
    {"label-value": 0, "rgba": [0, 0, 0, 255]},
    {"label-value": 1, "rgba": [255, 255, 255, 255]},
]

properties = [
    {"label-value": 0, "class": "background"},
    {"label-value": 1, "class": "foreground"},
]

We can now pass this metadata to the respective field of the {py:class}`ome_zarr.NgffMultiscales` object and write it to disk

```{note}
Under the hood, the `NgffMultiscales` class expects an instance of the {py:class}`ome_zarr_models.common.image_label_types.LabelBase` class. Hence, if the metadata is added to the image object after its creation, the `image_label` metadata needs to be wrapped in a `LabelBase` object before being passed to the `image_label` field of the `NgffMultiscales` class.
```

In [5]:
labels_multiscales.image_label = LabelBase.model_validate(
    {"colors": colors,
    "properties": properties},
)

ngff_multiscales.labels = {"test_labels": labels_multiscales}

# write to disk
ngff_multiscales.to_ome_zarr("labels_metadata_example.zarr")

[]

## Reading the image-label metadata

When reading an ome-zarr with labels from disk, the `image-label` metadata is automatically parsed and made available as part of the {py:class}`ome_zarr.NgffMultiscales` object.
We can retrieve the labels image from the `labels` attribute of the parent `NgffMultiscales` image:

In [6]:
image = NgffMultiscales.from_ome_zarr("labels_metadata_example.zarr")
image.labels["test_labels"].image_label

LabelBase(colors=(Color(label_value=0, rgba=(0, 0, 0, 255)), Color(label_value=1, rgba=(255, 255, 255, 255))), properties=(Property(label_value=0, class='background'), Property(label_value=1, class='foreground')), source=None, version='0.5')

This can be converted into a more human-readable format using the `model_dump()` method of the `LabelBase` class:

In [7]:
image.labels["test_labels"].image_label.model_dump()

{'colors': ({'label_value': 0, 'rgba': (0, 0, 0, 255)},
  {'label_value': 1, 'rgba': (255, 255, 255, 255)}),
 'properties': ({'label_value': 0, 'class': 'background'},
  {'label_value': 1, 'class': 'foreground'}),
 'source': None,
 'version': '0.5'}

Alternatively, it is possible to read only the label image without ingesting the parent image and its metadata:

In [8]:
label_image = image = NgffMultiscales.from_ome_zarr("labels_metadata_example.zarr/labels/test_labels")
label_image.image_label.model_dump()

{'colors': ({'label_value': 0, 'rgba': (0, 0, 0, 255)},
  {'label_value': 1, 'rgba': (255, 255, 255, 255)}),
 'properties': ({'label_value': 0, 'class': 'background'},
  {'label_value': 1, 'class': 'foreground'}),
 'source': None,
 'version': '0.5'}